In [8]:
import pandas as pd
import numpy as np

# --- Parameters ---
gas_price = 5

# Unit 1 (Base)
hr1 = 10.275
vom1 = 5
mc1 = round(hr1 * gas_price + vom1, 3)  # 56.375

# Unit 2 (Incremental)
hr2 = 6.839
vom2 = 2
mc2 = round(hr2 * gas_price + vom2, 3)  # 36.195

# --- Generate 11 evenly spaced MW points ---
unit1_mw = np.linspace(57, 190, 11)     # Unit 1 range
unit2_mw = np.linspace(40, 420, 11)
# --- Build table ---
df = pd.DataFrame({
    "Bid": range(1, 12),
    "Unit 1 MW": np.round(unit1_mw, 1),
    "Unit 1 ($/MWh)": [mc1]*11,
    "Unit 2 MW": np.round(unit2_mw, 1),
    "Unit 2 ($/MWh)": [mc2]*11
})

print(df)

    Bid  Unit 1 MW  Unit 1 ($/MWh)  Unit 2 MW  Unit 2 ($/MWh)
0     1       57.0          56.375       40.0          36.195
1     2       70.3          56.375       78.0          36.195
2     3       83.6          56.375      116.0          36.195
3     4       96.9          56.375      154.0          36.195
4     5      110.2          56.375      192.0          36.195
5     6      123.5          56.375      230.0          36.195
6     7      136.8          56.375      268.0          36.195
7     8      150.1          56.375      306.0          36.195
8     9      163.4          56.375      344.0          36.195
9    10      176.7          56.375      382.0          36.195
10   11      190.0          56.375      420.0          36.195


In [9]:
prices = pd.read_excel('Prices.xlsx')
prices.head()



,OPERATING_DATE,HOUR_ENDING,NP15 ($/MWh)
0,2022-03-21,1,45.04
1,2022-03-21,2,43.63
2,2022-03-21,3,43.43
3,2022-03-21,4,43.42
4,2022-03-21,5,46.30


In [ ]:
# =============================================================================
# TASK 3b: Two Independent Pseudo-Units - Separate Optimizations
# =============================================================================

import os
os.environ["GRB_LICENSE_FILE"] = os.path.abspath("gurobi.lic")

import gurobipy as gp
from gurobipy import GRB
import matplotlib.pyplot as plt

# --- Parameters ---
T = 168
gas_price = 5  # $/MMBtu

# Unit 1 (Base Unit)
min_mw_1, max_mw_1 = 57, 190
heat_rate_1, vom_1 = 10.275, 5
startup_cost_1 = 7250

# Unit 2 (Incremental Unit)
min_mw_2, max_mw_2 = 0, 420
heat_rate_2, vom_2 = 6.839, 2
startup_cost_2 = 16500

# Get electricity prices
price_col = [c for c in prices.columns if "NP15" in c][0]
price = prices[price_col].values[:T]


def optimize_unit(price, min_mw, max_mw, heat_rate, vom, startup_cost, gas_price, unit_name):
    """Optimize a single pseudo-unit independently. Returns p_val (MW per hour)."""
    model = gp.Model(f"Unit_{unit_name}")
    model.setParam("OutputFlag", 0)

    T_range = range(len(price))

    u = model.addVars(T_range, vtype=GRB.BINARY, name="u")
    p = model.addVars(T_range, vtype=GRB.CONTINUOUS, lb=0, name="p")
    v = model.addVars(T_range, vtype=GRB.BINARY, name="v")

    # Generation bounds
    for t in T_range:
        model.addConstr(p[t] >= min_mw * u[t])
        model.addConstr(p[t] <= max_mw * u[t])

    # Startup logic
    for t in T_range:
        if t == 0:
            model.addConstr(v[t] == u[t])
        else:
            model.addConstr(v[t] >= u[t] - u[t - 1])
            model.addConstr(v[t] <= u[t])

    # Objective: maximize profit
    revenue = gp.quicksum(price[t] * p[t] for t in T_range)
    cost = gp.quicksum(
        (heat_rate * gas_price + vom) * p[t] + startup_cost * v[t]
        for t in T_range
    )
    model.setObjective(revenue - cost, GRB.MAXIMIZE)
    model.optimize()

    p_val = [p[t].X for t in T_range]
    u_val = [u[t].X for t in T_range]
    v_val = [v[t].X for t in T_range]

    return p_val, u_val, v_val


# Optimize each unit separately
p1, u1, v1 = optimize_unit(
    price, min_mw_1, max_mw_1, heat_rate_1, vom_1, startup_cost_1, gas_price, "1"
)
p2, u2, v2 = optimize_unit(
    price, min_mw_2, max_mw_2, heat_rate_2, vom_2, startup_cost_2, gas_price, "2"
)

In [ ]:
# --- OUTPUT 1 & 2: Create DataFrame, save CSV, print first 12 rows ---
output_df = prices[["OPERATING_DATE", "HOUR_ENDING"]].iloc[:T].copy()
output_df["PRICE_ELECTRIC"] = price
output_df["PRICE_GAS"] = gas_price
output_df["MW_GENERATION_Unit1"] = p1
output_df["MW_GENERATION_Unit2"] = p2

output_df.to_csv("CCGT_PSEUDO.csv", index=False)

print("First 12 rows of CCGT_PSEUDO.csv:")
print(output_df.head(12).to_string())

In [ ]:
# --- OUTPUT 3: Plot MW vs time (both units) ---
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(T), p1, label="Unit 1", color="steelblue", linewidth=1.5)
ax.plot(range(T), p2, label="Unit 2", color="darkorange", linewidth=1.5)
ax.set_xlabel("Hour")
ax.set_ylabel("MW")
ax.set_title("MW Generation vs Time (Pseudo-Units)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- OUTPUT 4: Metrics for each unit ---
def compute_metrics(p_val, u_val, v_val, price, heat_rate, vom, startup_cost, gas_price, max_mw, unit_name):
    total_revenue = sum(price[t] * p_val[t] for t in range(T))
    total_var_cost = sum((heat_rate * gas_price + vom) * p_val[t] for t in range(T))
    total_startup = sum(startup_cost * v_val[t] for t in range(T))
    total_cost = total_var_cost + total_startup
    fuel_costs = sum(heat_rate * gas_price * p_val[t] for t in range(T))
    num_starts = sum(int(round(v_val[t])) for t in range(T))
    capacity_factor = 100 * sum(p_val) / (max_mw * T) if max_mw > 0 else 0
    gross_margin = (total_revenue - total_cost) / (max_mw * 1000) if max_mw > 0 else 0

    print(f"\n--- Unit {unit_name} ---")
    print(f"Total Revenue ($):     {total_revenue:,.2f}")
    print(f"Total Costs ($):       {total_cost:,.2f}")
    print(f"Fuel Costs ($):        {fuel_costs:,.2f}")
    print(f"Number of Starts:      {num_starts}")
    print(f"Capacity Factor (%):   {capacity_factor:.2f}%")
    print(f"Gross Margin ($/kW):   {gross_margin:.4f}")


print("=" * 50)
print("Pseudo-Unit Optimization Summary")
print("=" * 50)
compute_metrics(p1, u1, v1, price, heat_rate_1, vom_1, startup_cost_1, gas_price, max_mw_1, "1")
compute_metrics(p2, u2, v2, price, heat_rate_2, vom_2, startup_cost_2, gas_price, max_mw_2, "2")